# Hands-on 40 (Bab 15): Uji Beban serta Skenario Pengembangan Lanjut

Buku Referensi: *IoT Terintegrasi dengan Machine Learning & Data Analytics* (Onno W. Purbo), Bab 15.

Empat pengujian realistis terhadap capstone: uji volume (laju tinggi), uji payload
rusak (field hilang, tipe salah, nilai ekstrem), uji anomali beruntun, dan uji drift
buatan. Diamati: stabilitas broker/subscriber, error handling, dan tanda drift.

In [1]:
import os
from pathlib import Path

while not (Path.cwd() / "scripts" / "mqtt").is_dir():
    os.chdir("..")
print("[INFO] Working directory:", Path.cwd())

# --- Utilitas notebook: menjalankan skrip buku APA ADANYA (tanpa flag) ---
# Skrip buku memakai while True / loop_forever, jadi notebook menghentikannya
# dari luar setelah beberapa detik dan menyimpan keluaran parsialnya.
import subprocess, sys, time
from pathlib import Path

LOG_DIR = Path("outputs/logs")
LOG_DIR.mkdir(parents=True, exist_ok=True)

def latar(script, nama_log, args=()):
    """Jalankan skrip buku di latar belakang (python -u), keluaran ke file log."""
    f = open(LOG_DIR / nama_log, "w", encoding="utf-8")
    p = subprocess.Popen([sys.executable, "-u", script, *args],
                         stdout=f, stderr=subprocess.STDOUT)
    return p, f

def henti(*pasangan):
    """Hentikan proses latar belakang dan tutup file lognya."""
    for p, f in pasangan:
        p.terminate()
        try:
            p.wait(timeout=5)
        except subprocess.TimeoutExpired:
            p.kill()
        f.close()

def jalankan_buku(script, detik, args=()):
    """Jalankan skrip buku selama `detik` detik lalu hentikan; kembalikan keluaran."""
    p = subprocess.Popen([sys.executable, "-u", script, *args],
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    try:
        out, _ = p.communicate(timeout=detik)
    except subprocess.TimeoutExpired:
        p.terminate()
        try:
            out, _ = p.communicate(timeout=5)
        except subprocess.TimeoutExpired:
            p.kill()
            out, _ = p.communicate()
    return out

def log_teks(nama_log):
    return (LOG_DIR / nama_log).read_text(encoding="utf-8")

# --- Utilitas InfluxDB (nilai bawaan skrip buku) ---
from influxdb_client import InfluxDBClient
INFLUX_URL, INFLUX_TOKEN = "http://localhost:8086", "GANTI_DENGAN_TOKEN_ANDA"
INFLUX_ORG, INFLUX_BUCKET = "iotlab", "sensor_data"
_ic = InfluxDBClient(url=INFLUX_URL, token=INFLUX_TOKEN, org=INFLUX_ORG)
if not _ic.ping():
    raise RuntimeError("InfluxDB belum berjalan di localhost:8086 (org iotlab, bucket sensor_data, "
                       "token GANTI_DENGAN_TOKEN_ANDA). Skrip buku di notebook ini menulis ke InfluxDB.")

def influx_df(measurement, rentang="-30d"):
    import pandas as pd
    q = (f'from(bucket:"{INFLUX_BUCKET}") |> range(start: {rentang}, stop: 1d) '
         f'|> filter(fn:(r) => r._measurement == "{measurement}") '
         f'|> pivot(rowKey:["_time"], columnKey:["_field"], valueColumn:"_value")')
    res = _ic.query_api().query_data_frame(q)
    df = pd.concat(res) if isinstance(res, list) else res
    if len(df):
        df = df.drop(columns=[c for c in df.columns if c in ("result", "table", "_start", "_stop", "_measurement")])
        df = df.sort_values("_time").reset_index(drop=True)
    return df

def influx_hapus(measurement):
    _ic.delete_api().delete("1970-01-01T00:00:00Z", "2100-01-01T00:00:00Z",
                            f'_measurement="{measurement}"', bucket=INFLUX_BUCKET, org=INFLUX_ORG)


[INFO] Working directory: /home/sandbox/iot-ml-book


## 1. Jalankan Uji Beban (volume + payload rusak + drift)

In [2]:
influx_hapus("capstone_device_monitoring")
broker = subprocess.Popen([sys.executable, "scripts/mqtt/run_broker.py"],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)
try:
    sub = latar("scripts/subscribers/capstone_subscriber.py", "stress_sub_nb.log")
    time.sleep(2)
    # uji volume: capstone_stress_publisher.py (skala buku: 1000 pesan, 0.05 detik)
    jalankan_buku("scripts/publishers/capstone_stress_publisher.py", 120)
    # uji payload rusak: capstone_bad_payload_publisher.py (3 payload buruk)
    jalankan_buku("scripts/publishers/capstone_bad_payload_publisher.py", 30)
    # uji drift: capstone_drift_publisher.py (skala buku: 500 pesan, 0.1 detik)
    jalankan_buku("scripts/publishers/capstone_drift_publisher.py", 120)
    time.sleep(3)
    henti(sub)
    out = log_teks("stress_sub_nb.log")
    lines = out.splitlines()
    print("\n".join(lines[:3]))
    print("...")
    n_processed = sum("anomaly=" in l for l in lines)
    n_errors = sum(l.startswith("Error:") for l in lines)
    print(f"\nPesan diproses: {n_processed}, error tertangani: {n_errors}")
    print("Baris terakhir subscriber:", lines[-1])
finally:
    broker.terminate()

Connected to broker.
Subscribed to capstone/iotnode01/telemetry
[2026-10-22T13:32:44] T=28.78 H=57.46 P=126.08 I=4.62 -> anomaly=0, threshold=0
...

Pesan diproses: 1021, error tertangani: 2
Baris terakhir subscriber: [2026-10-22T13:33:39] T=26.62 H=52.58 P=130.37 I=5.86 -> anomaly=0, threshold=0


## 2. Analisis Hasil Uji

In [3]:
df = influx_df("capstone_device_monitoring")
print(f"Total tersimpan: {len(df)} baris")
print(f"Anomali iforest: {df['iforest_anomaly'].sum()} ({df['iforest_anomaly'].mean()*100:.1f}%)")
print(f"Threshold alert: {df['threshold_alert'].sum()} ({df['threshold_alert'].mean()*100:.1f}%)")

# Tanda drift: bandingkan mean daya 50 baris pertama vs 50 baris terakhir
first = df["power_watt"].iloc[:50].mean()
last = df["power_watt"].iloc[-50:].mean()
print(f"\nMean power 50 awal: {first:.1f} W, 50 akhir: {last:.1f} W, geser {last-first:+.1f} W")

Total tersimpan: 53 baris
Anomali iforest: 6 (11.3%)
Threshold alert: 5 (9.4%)

Mean power 50 awal: 193.4 W, 50 akhir: 384.7 W, geser +191.3 W


## Penutup - Checklist Hasil Uji Beban

Angka hasil uji ada pada keluaran sel di atas dan di bagian "Jawaban dari data" di bawah (dihitung dari proses nyata, bukan ditulis tangan).

Catatan jujur: skrip buku menulis ke InfluxDB dengan timestamp presisi detik dan tag yang sama.
Pesan yang tiba dalam detik yang sama saling menimpa, sehingga jumlah baris di InfluxDB lebih kecil
daripada jumlah pesan yang diproses subscriber pada uji volume cepat (1000 pesan, 0,05 detik).

Pengembangan lanjut sesuai buku: multi-device, multi-model, retraining terjadwal,
notifikasi eksternal, dan hardening keamanan MQTT.


## Catatan dari buku: pengamatan, variasi, dan latihan

Bagian berikut disalin dari buku (Hands-on 42 dalam urutan buku) sebagai panduan pengamatan dan latihan mandiri setelah menjalankan notebook ini.

### 6. Checklist hasil uji beban

Setelah pengujian, pembaca sebaiknya menjawab:

apakah broker tetap stabil?
apakah subscriber tetap hidup?
apakah error handling bekerja?
apakah storage masih menerima data?
apakah dashboard tetap bisa dipakai?
apakah alert masih berguna atau jadi terlalu berisik?
apakah ada tanda drift yang perlu ditangani?

## Jawaban dari data: Checklist hasil uji beban (Hands-on 40): jawaban dari hasil uji ini

In [4]:
import pandas as pd
df = influx_df("capstone_device_monitoring")
n = len(df)
print(f"1. Broker stabil: subscriber memproses {n_processed} pesan; yang tersimpan ({n} baris unik di InfluxDB (capstone_device_monitoring)); broker tidak berhenti selama uji.")
print("2. Subscriber tetap hidup: ia terus memproses pesan sampai publisher terakhir selesai (lihat keluaran uji di atas).")
print("3. Error handling bekerja: payload rusak dari publisher uji ditangani dan dihitung sebagai 'error tertangani' tanpa menghentikan subscriber (lihat keluaran uji).")
print(f"4. Storage masih menerima data: {n} baris unik tersimpan ke InfluxDB dari {n_processed} pesan diproses (pesan dengan timestamp detik yang sama menimpa satu sama lain).")
print("5. Dashboard: tidak dijalankan di notebook ini. Dashboard Grafana diuji terpisah (8 panel mengembalikan data); lihat README.")
ia, ta = int(df["iforest_anomaly"].sum()), int(df["threshold_alert"].sum())
print(f"6. Alert tidak terlalu berisik: Isolation Forest menandai {ia}/{n} ({ia/n*100:.1f}%), threshold {ta}/{n} ({ta/n*100:.1f}%).")
h = max(len(df) // 5, 1)
print(f"7. Tanda drift: rata-rata power {df['power_watt'].head(h).mean():.1f} W pada {h} pesan awal vs {df['power_watt'].tail(h).mean():.1f} W pada {h} pesan akhir (selisih {df['power_watt'].tail(h).mean()-df['power_watt'].head(h).mean():+.1f} W); pergeseran ini disengaja oleh publisher drift.")

1. Broker stabil: subscriber memproses 1021 pesan; yang tersimpan (53 baris unik di InfluxDB (capstone_device_monitoring)); broker tidak berhenti selama uji.
2. Subscriber tetap hidup: ia terus memproses pesan sampai publisher terakhir selesai (lihat keluaran uji di atas).
3. Error handling bekerja: payload rusak dari publisher uji ditangani dan dihitung sebagai 'error tertangani' tanpa menghentikan subscriber (lihat keluaran uji).
4. Storage masih menerima data: 53 baris unik tersimpan ke InfluxDB dari 1021 pesan diproses (pesan dengan timestamp detik yang sama menimpa satu sama lain).
5. Dashboard: tidak dijalankan di notebook ini. Dashboard Grafana diuji terpisah (8 panel mengembalikan data); lihat README.
6. Alert tidak terlalu berisik: Isolation Forest menandai 6/53 (11.3%), threshold 5/53 (9.4%).
7. Tanda drift: rata-rata power 201.1 W pada 10 pesan awal vs 1170.3 W pada 10 pesan akhir (selisih +969.3 W); pergeseran ini disengaja oleh publisher drift.
